In [6]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import List, Tuple, Dict, Iterable

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# ----------------------------
# 0) Reproducibility controls
# ----------------------------
BASE_SEED = 42
tf.keras.utils.set_random_seed(BASE_SEED)
np.random.seed(BASE_SEED)

In [7]:
# ----------------------------
# 1) Data Loading (kept similar)
# ----------------------------
def load_spy_rv(csv_path: str) -> pd.DataFrame:
    """
    Load SPY realized volatility as provided, keep structure minimal.
    Expected columns: Date, Volatility, Type; filter Type=='QMLE-Trade'.
    Index is a DatetimeIndex named 'date'. Returns a sorted DataFrame with a single
    column 'RV_daily' (float), indexed by trading date.
    """
    rv = pd.read_csv(csv_path)
    rv = rv[["Date", "Volatility", "Type"]]
    rv.rename(columns={"Volatility": "RV_daily"}, inplace=True)
    rv = rv[rv['Type'] == 'QMLE-Trade']
    rv.drop(columns=['Type'], inplace=True)
    rv = rv.set_index("Date")
    rv.index = pd.to_datetime(rv.index)
    rv.index.name = "date"
    rv = rv.sort_index()
    # ensure float
    rv["RV_daily"] = rv["RV_daily"].astype(float)
    return rv

In [8]:
# ----------------------------
# 2) Feature engineering
# ----------------------------
def add_lag_features(df: pd.DataFrame, base_lags: Iterable[int],
                     extra_lags: Iterable[int] = (132, 264)) -> Tuple[pd.DataFrame, List[int]]:
    """
    Create lag features from provided candidate set with optional longer lags.
    Returns DataFrame with 'RV_daily' and lag columns, and the final sorted lag list used.
    """
    # Deduplicate and sort lags
    all_lags = sorted(set(list(base_lags) + list(extra_lags)))
    out = df.copy()
    for L in all_lags:
        out[f"lag_{L}"] = out["RV_daily"].shift(L)

    # Drop rows with missing lags
    out = out.dropna()
    # Determine which lags actually survived (in case data too short)
    used_lags = [int(col.split("_")[1]) for col in out.columns if col.startswith("lag_")]
    used_lags = sorted(used_lags)
    return out, used_lags

def make_lag_matrix(out: pd.DataFrame, used_lags: List[int]) -> Tuple[np.ndarray, np.ndarray, pd.DatetimeIndex]:
    """
    Returns:
      X_2d: shape (N, n_lags) with columns ordered by ascending lag (largest to smallest or vice versa).
      y: shape (N,)
      idx: DatetimeIndex for samples
    We'll order timesteps from *largest lag to smallest lag* so the last step is the most recent (lag=1).
    """
    # Order largest -> smallest so final timestep corresponds to lag=1 (most recent)
    lag_cols = [f"lag_{L}" for L in sorted(used_lags, reverse=True)]
    X_2d = out[lag_cols].values.astype(np.float32)
    y = out["RV_daily"].values.astype(np.float32)
    idx = out.index
    return X_2d, y, idx

def zscore_fit_transform_train(X_train_2d: np.ndarray) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
    """
    Fit z-score scaler on training X (2D). Return scaled train and scaler parameters.
    (We scale only the inputs; the target y stays on original scale to preserve positivity semantics.)
    """
    mean = X_train_2d.mean(axis=0, keepdims=True)
    std = X_train_2d.std(axis=0, keepdims=True)
    std = np.where(std < 1e-12, 1.0, std)
    Xs = (X_train_2d - mean) / std
    return Xs, {"mean": mean, "std": std}

def zscore_apply(X_2d: np.ndarray, scaler: Dict[str, np.ndarray]) -> np.ndarray:
    return (X_2d - scaler["mean"]) / scaler["std"]

def to_lstm_sequence(X_2d: np.ndarray) -> np.ndarray:
    """
    Reshape from (N, n_lags) -> (N, n_steps=n_lags, n_features=1)
    """
    return X_2d[..., None]  # add feature dimension


In [9]:
# ----------------------------
# 3) Time splits (expanding window)
# ----------------------------
@dataclass
class Fold:
    train_start: pd.Timestamp
    train_end: pd.Timestamp
    val_start: pd.Timestamp
    val_end: pd.Timestamp
    test_start: pd.Timestamp
    test_end: pd.Timestamp

def month_start_after(date: pd.Timestamp) -> pd.Timestamp:
    # first day of next month
    return (date + pd.offsets.MonthBegin(1)).normalize()

def month_end_for(date: pd.Timestamp) -> pd.Timestamp:
    # last day of that month
    return (date + pd.offsets.MonthEnd(0)).normalize()

def generate_expanding_folds(index: pd.DatetimeIndex,
                             initial_train_years: int = 3,
                             val_years: int = 1) -> List[Fold]:
    """
    Produce folds where:
      - Training: from first available date to start of validation - 1 day (expands over time)
      - Validation: 1-year window immediately before test month
      - Test: the next calendar month after training+validation
    The very first fold will have approximately 3y train + 1y val.
    """
    if len(index) == 0:
        return []

    first = index.min()
    last = index.max()

    # Initial 4 years (3 train + 1 val) end date:
    min_val_end = first + pd.DateOffset(years=initial_train_years + val_years) - pd.Timedelta(days=1)
    # Start with the next month as initial test month
    test_start = month_start_after(min_val_end)
    folds: List[Fold] = []

    while True:
        test_end = month_end_for(test_start)
        if test_start > last:
            break

        # Actual available dates may be sparser; still define by calendar
        val_end = (test_start - pd.Timedelta(days=1))
        val_start = val_end - pd.DateOffset(years=val_years) + pd.Timedelta(days=1)

        # Training expands from the first date of index up to day before val_start
        train_start = first
        train_end = val_start - pd.Timedelta(days=1)

        # Stop if windows are invalid or beyond data:
        if train_end < train_start:
            break

        folds.append(Fold(train_start, train_end, val_start, val_end, test_start, test_end))

        # Move to the next month
        test_start = month_start_after(test_end)

    return folds

In [10]:
# ----------------------------
# 4) Model & training
# ----------------------------
def build_lstm_model(n_steps: int,
                     lr: float = 1e-3) -> tf.keras.Model:
    """
    2 hidden LSTM layers with 32 and 16 units, BatchNorm between layers,
    softplus output to ensure positivity.
    """
    inp = layers.Input(shape=(n_steps, 1))
    x = layers.LSTM(32, return_sequences=True)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.LSTM(16, return_sequences=False)(x)
    x = layers.BatchNormalization()(x)
    # Positive outputs
    out = layers.Dense(1, activation='softplus')(x)

    model = models.Model(inputs=inp, outputs=out)
    opt = optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='mse')
    return model

def qlike(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-12) -> float:
    """
    QLIKE = (y/f) - log(y/f) - 1 ; defined for y>0, f>0.
    """
    y = np.clip(y_true, eps, None)
    f = np.clip(y_pred, eps, None)
    ratio = y / f
    return float(np.mean(ratio - np.log(ratio) - 1.0))

def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((y_true - y_pred) ** 2))

def train_ensemble(X_tr, y_tr, X_val, y_val, X_te,
                   n_steps: int,
                   n_ensembles: int = 10,
                   lr: float = 1e-3,
                   batch_size: int = 1024,
                   epochs: int = 100,
                   patience: int = 10,
                   verbose: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    """
    Train N separate models with different seeds and average predictions.
    Returns:
      (train_pred_avg, test_pred_avg)
    """
    train_preds_accum = np.zeros((X_tr.shape[0],), dtype=np.float64)
    test_preds_accum = np.zeros((X_te.shape[0],), dtype=np.float64)

    for k in range(n_ensembles):
        tf.keras.utils.set_random_seed(BASE_SEED + k + 1)
        np.random.seed(BASE_SEED + k + 1)

        model = build_lstm_model(n_steps=n_steps, lr=lr)
        es = callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=False,  # time-series
            callbacks=[es],
            verbose=verbose
        )

        train_preds_accum += model.predict(X_tr, batch_size=batch_size, verbose=0).ravel()
        test_preds_accum  += model.predict(X_te, batch_size=batch_size, verbose=0).ravel()

    train_pred_avg = (train_preds_accum / n_ensembles).astype(np.float32)
    test_pred_avg  = (test_preds_accum  / n_ensembles).astype(np.float32)
    return train_pred_avg, test_pred_avg

In [12]:
# ----------------------------
# 5) Orchestration
# ----------------------------
def slice_by_date(idx: pd.DatetimeIndex, start: pd.Timestamp, end: pd.Timestamp) -> np.ndarray:
    """
    Return boolean mask for idx between [start, end] inclusive.
    """
    return (idx >= start) & (idx <= end)

def run_pipeline(
    csv_path: str = "SPY.csv",
    ticker: str = "SPY",
    base_lags: Iterable[int] = (1, 5, 16, 22, 32, 44, 66, 88),
    add_longer_lags: bool = True,
    longer_lags: Iterable[int] = (132, 264),
    learning_rate: float = 1e-3,
    patience: int = 10,
    n_ensembles: int = 10,
    batch_size: int = 1024,
    epochs: int = 100,
    plot_path: str = "lstm_oos_plot.png",
    preds_csv: str = "lstm_results_SPY.csv",
    metrics_csv: str = "lstm_metrics_per_fold.csv",
    verbose: int = 0
):
    # 1) Load data
    df = load_spy_rv(csv_path)

    # Optional: filter sample range if desired, e.g. ['1996-01-01', '2025-04-30']
    # df = df.loc["1996-01-01":"2025-04-30"]

    # 2) Lags
    if add_longer_lags:
        out, used_lags = add_lag_features(df, base_lags=base_lags, extra_lags=longer_lags)
    else:
        out, used_lags = add_lag_features(df, base_lags=base_lags, extra_lags=())
    if len(used_lags) == 0:
        raise ValueError("No usable lags after shifting; check data length.")

    # 3) Matrix + index
    X_all_2d, y_all, all_idx = make_lag_matrix(out, used_lags)
    n_steps = X_all_2d.shape[1]

    # 4) Folds
    folds = generate_expanding_folds(all_idx, initial_train_years=3, val_years=1)
    if len(folds) == 0:
        raise ValueError("No valid expanding folds produced. Check date coverage.")

    # 5) Storage for outputs
    test_rows = []   # For OOS predictions CSV
    metrics_rows = []  # Per-fold metrics

    # 6) Iterate folds
    for i, f in enumerate(folds, start=1):
        # Restrict to available dates actually present in our (lagged) index
        tr_mask = slice_by_date(all_idx, f.train_start, f.train_end)
        va_mask = slice_by_date(all_idx, f.val_start,   f.val_end)
        te_mask = slice_by_date(all_idx, f.test_start,  f.test_end)

        # Skip folds where test has zero observations (e.g., very last month may be incomplete)
        if te_mask.sum() == 0 or va_mask.sum() == 0 or tr_mask.sum() == 0:
            continue

        X_tr_2d, y_tr = X_all_2d[tr_mask], y_all[tr_mask]
        X_va_2d, y_va = X_all_2d[va_mask], y_all[va_mask]
        X_te_2d, y_te = X_all_2d[te_mask], y_all[te_mask]
        idx_te = all_idx[te_mask]
        idx_tr = all_idx[tr_mask]

        # Scale inputs using TRAIN ONLY
        X_tr_scaled, scaler = zscore_fit_transform_train(X_tr_2d)
        X_va_scaled = zscore_apply(X_va_2d, scaler)
        X_te_scaled = zscore_apply(X_te_2d, scaler)

        # Reshape to LSTM sequences
        X_tr = to_lstm_sequence(X_tr_scaled)  # (N_tr, n_steps, 1)
        X_va = to_lstm_sequence(X_va_scaled)
        X_te = to_lstm_sequence(X_te_scaled)

        # Train ensemble & predict
        tr_pred, te_pred = train_ensemble(
            X_tr, y_tr, X_va, y_va, X_te,
            n_steps=n_steps,
            n_ensembles=n_ensembles,
            lr=learning_rate,
            batch_size=batch_size,
            epochs=epochs,
            patience=patience,
            verbose=verbose
        )

        # Metrics (in-sample = training years; OOS = test month)
        ins_mse = mse(y_tr, tr_pred)
        ins_qlk = qlike(y_tr, tr_pred)
        oos_mse = mse(y_te, te_pred)
        oos_qlk = qlike(y_te, te_pred)

        metrics_rows.append({
            "fold": i,
            "train_start": f.train_start.date(),
            "train_end":   f.train_end.date(),
            "val_start":   f.val_start.date(),
            "val_end":     f.val_end.date(),
            "test_start":  f.test_start.date(),
            "test_end":    f.test_end.date(),
            "train_obs":   int(len(y_tr)),
            "val_obs":     int(len(y_va)),
            "test_obs":    int(len(y_te)),
            "insample_MSE": ins_mse,
            "insample_QLIKE": ins_qlk,
            "oos_MSE": oos_mse,
            "oos_QLIKE": oos_qlk
        })

        # Store OOS predictions
        for d, y_true, y_hat in zip(idx_te, y_te, te_pred):
            test_rows.append({
                "ticker": ticker,
                "date": d.date(),
                "rv_real": float(y_true),
                "rv_pred": float(y_hat)
            })

        # (Optional) print concise progress
        print(f"[Fold {i:03d}] "
              f"Train {f.train_start.date()}→{f.train_end.date()} | "
              f"Val {f.val_start.date()}→{f.val_end.date()} | "
              f"Test {f.test_start.date()}→{f.test_end.date()} | "
              f"OOS MSE={oos_mse:.6g}, QLIKE={oos_qlk:.6g}")

    # 7) Save outputs
    test_df = pd.DataFrame(test_rows).sort_values("date").reset_index(drop=True)
    metrics_df = pd.DataFrame(metrics_rows)

    test_df.to_csv(preds_csv, index=False)
    metrics_df.to_csv(metrics_csv, index=False)

    # 8) Summary metrics across folds
    # Simple (unweighted) average across folds:
    avg_ins_mse  = metrics_df["insample_MSE"].mean()
    avg_ins_qlk  = metrics_df["insample_QLIKE"].mean()
    avg_oos_mse  = metrics_df["oos_MSE"].mean()
    avg_oos_qlk  = metrics_df["oos_QLIKE"].mean()

    print("\n=== Summary Across Folds ===")
    print(f"Insample (training years)  : MSE={avg_ins_mse:.6g} | QLIKE={avg_ins_qlk:.6g}")
    print(f"Out-of-sample (test month) : MSE={avg_oos_mse:.6g} | QLIKE={avg_oos_qlk:.6g}")
    print(f"Saved OOS predictions -> {os.path.abspath(preds_csv)}")
    print(f"Saved per-fold metrics -> {os.path.abspath(metrics_csv)}")

    # 9) Plot OOS actual vs predicted
    if not test_df.empty:
        plt.figure(figsize=(12, 5))
        plt.plot(pd.to_datetime(test_df["date"]), test_df["rv_real"], label="Actual RV (OOS)")
        plt.plot(pd.to_datetime(test_df["date"]), test_df["rv_pred"], label="Predicted RV (OOS)", alpha=0.85)
        plt.title(f"{ticker} Realized Volatility: Actual vs Predicted (OOS test months)")
        plt.xlabel("Date")
        plt.ylabel("Realized Volatility")
        plt.legend()
        plt.tight_layout()
        plt.savefig(plot_path, dpi=150)
        plt.show()
        print(f"Saved plot -> {os.path.abspath(plot_path)}")
    else:
        print("No OOS predictions to plot (empty test set).")

if __name__ == "__main__":
    run_pipeline(
        csv_path="SPY.csv",      # <-- keep your path/filename here
        ticker="SPY",
        base_lags=(1, 5, 16, 22, 32, 44, 66, 88),
        add_longer_lags=True,    # set False if you want only your original lags
        longer_lags=(132, 264),  # ~6m and ~12m
        learning_rate=1e-3,
        patience=10,
        n_ensembles=10,
        batch_size=1024,
        epochs=100,
        plot_path="lstm_oos_plot.png",
        preds_csv="lstm_results_SPY.csv",
        metrics_csv="lstm_metrics_per_fold.csv",
        verbose=0
    )

[Fold 001] Train 1997-01-17→2000-01-31 | Val 2000-02-01→2001-01-31 | Test 2001-02-01→2001-02-28 | OOS MSE=0.128714, QLIKE=0.509955
[Fold 002] Train 1997-01-17→2000-02-28 | Val 2000-02-29→2001-02-28 | Test 2001-03-01→2001-03-31 | OOS MSE=0.0833169, QLIKE=0.27156
[Fold 003] Train 1997-01-17→2000-03-31 | Val 2000-04-01→2001-03-31 | Test 2001-04-01→2001-04-30 | OOS MSE=0.0939944, QLIKE=0.32178
[Fold 004] Train 1997-01-17→2000-04-30 | Val 2000-05-01→2001-04-30 | Test 2001-05-01→2001-05-31 | OOS MSE=0.135154, QLIKE=0.528273
[Fold 005] Train 1997-01-17→2000-05-31 | Val 2000-06-01→2001-05-31 | Test 2001-06-01→2001-06-30 | OOS MSE=0.12841, QLIKE=0.55081
[Fold 006] Train 1997-01-17→2000-06-30 | Val 2000-07-01→2001-06-30 | Test 2001-07-01→2001-07-31 | OOS MSE=0.129647, QLIKE=0.559183
[Fold 007] Train 1997-01-17→2000-07-31 | Val 2000-08-01→2001-07-31 | Test 2001-08-01→2001-08-31 | OOS MSE=0.133092, QLIKE=0.584261
[Fold 008] Train 1997-01-17→2000-08-31 | Val 2000-09-01→2001-08-31 | Test 2001-09-01→

KeyboardInterrupt: 